In [ ]:
import dask
# dask.config.set(scheduler="synchronous")

from toolviper.dask.client import local_client

viper_client = local_client(cores=4, memory_limit="4GB")
viper_client

## Download Dataset

In [ ]:
from xradio.measurement_set import open_processing_set
from toolviper.utils.data import download, update
update()

download(file="twhya_selfcal_5chans_lsrk_compare_weights.ps.zarr")
ps_xdt = open_processing_set("twhya_selfcal_5chans_lsrk_compare_weights.ps.zarr")

download(file="twhya_selfcal_5chans_lsrk_xx_compare_weights.ps.zarr")
ps_single_pol_xdt = open_processing_set("twhya_selfcal_5chans_lsrk_xx_compare_weights.ps.zarr")


download(file="3c286_Band6_5chans_lsrk_compare_weights.ps.zarr")
ps_full_pol_xdt = open_processing_set("3c286_Band6_5chans_lsrk_compare_weights.ps.zarr")


ps_xdt.xr_ps.summary()

In [ ]:
ps_xdt["twhya_selfcal_5chans_lsrk_0"].attrs["data_groups"].keys()

In [1]:

%load_ext autoreload
%autoreload 2

import dask
dask.config.set(scheduler="synchronous")

# from toolviper.dask.client import local_client

# viper_client = local_client(cores=4, memory_limit="4GB")
# viper_client

log_level = "DEBUG" 
#log_level = "INFO" 
log_to_file = False
log_params = { "logger_name": "main",
            "log_to_term": True,
            "log_level": log_level,
            "log_to_file": log_to_file,
            "log_file": "client.log",
            }

worker_logs = { "logger_name": "worker",
            "log_to_term": True,
            "log_level": log_level,
            "log_to_file": log_to_file,
            "log_file": "client_worker.log",
            }


from toolviper.dask.client import local_client

viper_client = local_client(cores=1, memory_limit="18GB", log_params=log_params, worker_log_params=worker_logs)
viper_client

import os
import numpy as np
from xradio.measurement_set import open_processing_set
from astroviper.distributed.imaging.image_cube_single_field import image_cube_single_field
from xradio.image import make_empty_sky_image

os.system("rm -rf twhya_selfcal_5chans_lsrk_compare_weights.img.zarr")
ps_single_pol_xdt = open_processing_set("twhya_selfcal_5chans_lsrk_compare_weights.ps.zarr")
combined_field_and_source_xds = ps_single_pol_xdt.xr_ps.get_combined_field_and_source_xds()
center_field_name = combined_field_and_source_xds.attrs["center_field_name"]
phase_direction = (
    combined_field_and_source_xds.FIELD_PHASE_CENTER_DIRECTION.sel(
        field_name=center_field_name
    )
)

#print(ps_single_pol_xdt.xr_ps.get_freq_axis())

empty_img_xds = make_empty_sky_image(
        phase_center=phase_direction.values,
        image_size=[250,250],
        cell_size=np.array([-0.1,0.1]) * np.pi/(180 * 3600),
        frequency_coords=ps_single_pol_xdt.xr_ps.get_freq_axis().values,
        pol_coords=["I"],
        time_coords=[0],
    )

# image_cube_single_field

imaging_metadata_dict = image_cube_single_field(
    ps_store = "twhya_selfcal_5chans_lsrk_compare_weights.ps.zarr",
    image_store = "twhya_selfcal_5chans_lsrk_compare_weights.img.zarr",
    image_params={
        "image_size": [250, 250],
        "cell_size": np.array([0.1, 0.1]) * np.pi/(180 * 3600),
        "phase_direction": phase_direction.values,
        "frequency_coords": ps_single_pol_xdt.xr_ps.get_freq_axis().values,
        "polarization_coords": ["I","Q"],
        "time_coords": [0],
    },
    imaging_weights_params={
        "weighting": "briggs",
        "robust": 0.5,
    },
    # imaging_weights_params={
    #     "weighting": "natural",
    # },
    iteration_control_params={
        "niter": 0,
        "nmajor": 0,
        "threshold": 0.0,
        "gain": 0.1,
        "cyclefactor": 1.5,
        "cycleniter": 10,
    },
    gridder="prolate_spheroidal",
    deconvolver="hogbom",
    fft_padding="1.0",
    scan_intents="OBSERVE_TARGET#ON_SOURCE",
    #image_data_variables_keep=["sky", "point_spread_function", "primary_beam"],
    # image_data_variables_keep=["sky_model", "sky_residual", "sky_deconvolved", "point_spread_function", "primary_beam"],
    image_data_variables_keep=["sky_residual", "point_spread_function", "primary_beam"],
    #image_data_variables_keep=[ "sky", "point_spread_function", "primary_beam"],
    processing_set_data_group_name="base",
    double_precision=True,
    thread_info=None,
    n_chunks=None,
    overwrite=True,
)
imaging_metadata_dict

[2026-03-18 19:28:03,222]  WARNING        main:  It is recommended that the local cache directory be set using the dask_local_dir parameter. 
[2026-03-18 19:28:03,765]    DEBUG        main:  Loading plugin module: <class 'worker.DaskWorker'>
[2026-03-18 19:28:04,284]    DEBUG    worker_0:  Logger created on worker Worker-ae114afb-b0b7-4a75-a5ee-ff77cd2cb75c,*,tcp://127.0.0.1:60470
[2026-03-18 19:28:04,285]     INFO        main:  Client <MenrvaClient: 'tcp://127.0.0.1:60464' processes=1 threads=1, memory=16.76 GiB> 
[2026-03-18 19:28:05,737]     INFO        main:  Time to create empty image xds: 0.001255035400390625 seconds 
[2026-03-18 19:28:05,750]     INFO        main:  Time to write empty image to disk: 0.01292109489440918 seconds 


INFO:main:Time to write empty image to disk: 0.01292109489440918 seconds


[2026-03-18 19:28:05,751]     INFO        main:  Memory required for a single frequency channel: 0.005029141902923584 GiB 


INFO:main:Memory required for a single frequency channel: 0.005029141902923584 GiB


[2026-03-18 19:28:05,751]     INFO        main:  Thread info {'n_threads': 1, 'memory_per_thread': 16.763806343078613} 


...

[2026-03-18 19:28:08,296]    DEBUG    worker_0:  Memory usage after releasing references: 0.423215104 GB
[2026-03-18 19:28:08,303]    DEBUG    worker_0:  Memory usage at start of wrap_image_cube_single_field_node_task: 0.423215104 GB
[2026-03-18 19:28:08,304]    DEBUG    worker_0:  Processing set iterator created with partitions.
[2026-03-18 19:28:08,304]    DEBUG    worker_0:  Processing chunk 1
[2026-03-18 19:28:08,305]    DEBUG    worker_0:  img_xds size 4.108e-06 GB
[2026-03-18 19:28:08,343]    DEBUG    worker_0:  Calculate imaging weights 0.03781914710998535
[2026-03-18 19:28:08,351]    DEBUG    worker_0:  Warning: Overwriting data variables in existing data group single_field since overwrite=True.
##### Data groups in image dict_keys(['base', 'single_field'])
[2026-03-18 19:28:08,359]    DEBUG    worker_0:  Warning: Overwriting data variables in existing data group single_field since overwrite=True.
[2026-03-18 19:28:08,366]    DEBUG    worker_0:  Load data and add to grid 0.0338

INFO:main:Time to compute dask graph: 2.650545120239258 seconds


[2026-03-18 19:28:08,509]     INFO        main:  Time to consolidate metadata: 0.007193088531494141 seconds 


INFO:main:Time to consolidate metadata: 0.007193088531494141 seconds


,T_weights,T_vis_mask,T_uv_sampling_grid,T_vis_grid,T_add_to_grid,T_fft_norm,T_load,task_id,n_channels
0,0.003937,0.007661,0.005136,0.005501,0.018307,0.009640,0.030220,0,2
1,0.003957,0.008264,0.007572,0.007642,0.023494,0.016363,0.033861,1,2
2,0.153293,0.007069,0.007027,0.007059,0.021171,1.111090,0.037666,2,1


In [ ]:
import xarray as xr
img_xds = xr.open_zarr("twhya_selfcal_5chans_lsrk_compare_weights.img.zarr")
img_xds

In [ ]:
from astroviper.core.imaging.imaging_utils.gcf_prolate_spheroidal import create_prolate_spheroidal_kernel
kernel, kernel_image_1D_u, kernel_image_1D_v = create_prolate_spheroidal_kernel(100, 7, n_uv=[250,250])

import matplotlib.pyplot as plt
kernel_image = np.outer(kernel_image_1D_u, kernel_image_1D_v)
plt.figure(figsize=(10,10))
plt.imshow(kernel[:,:,3,3])
plt.colorbar()
plt.show()


In [ ]:
import xarray as xr
img_xds = xr.open_zarr("twhya_selfcal_5chans_lsrk_compare_weights.img.zarr")
img_xds

import matplotlib.pyplot as plt
plt.imshow(img_xds.PRIMARY_BEAM.isel(frequency=0, polarization=1, time=0).values)
plt.colorbar()

# import matplotlib.pyplot as plt
# plt.imshow(img_xds.POINT_SPREAD_FUNCTION.isel(frequency=0, polarization=1, time=0).values)
# plt.colorbar()

import matplotlib.pyplot as plt
plt.imshow(img_xds.SKY_RESIDUAL.isel(frequency=3, polarization=0, time=0).values)
plt.colorbar()